# Zero-Shot, One-Shot, and Few-Shot Prompting with Groq

**Goal:** Ask the same sentiment-analysis question using three prompting techniques and compare the responses.

**Shared question:**  
`The hotel room was clean and the staff were friendly, but the wifi barely worked.`

## Setup
1. Install dependencies in your environment: `pip install groq python-dotenv openai`
2. Add your Groq API key to a `.env` file as `GROQ_API_KEY=your_actual_groq_api_key`.
3. Run the notebook cells from top to bottom.

Keep API keys private. Do not paste them directly into notebook code or share the `.env` file.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY was not found. Add it to your .env file and restart the notebook kernel."
    )

# Groq provides an OpenAI-compatible Chat Completions API.
client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)

# Change this to a model currently available to your Groq account if necessary.
MODEL = "openai/gpt-oss-20b"

user_query = (
    "The hotel room was clean and the staff were friendly, "
    "but the wifi barely worked."
)

system_prompt = """You analyze sentiment.
Respond only in XML using exactly this structure:
<sentiment>
  <label>positive|negative|neutral|mixed</label>
  <confidence>0.0-1.0</confidence>
</sentiment>
Do not include any text outside the XML.
Use 'mixed' when the text contains both positive and negative sentiment."""

print("Groq client initialized.")
print("Model:", MODEL)
print("Shared question:", user_query)

Groq client initialized.
Model: openai/gpt-oss-20b
Shared question: The hotel room was clean and the staff were friendly, but the wifi barely worked.


## 1. Zero-Shot Prompting

**Zero-shot** means the model receives instructions and the question, but no examples.

In [2]:
zero_shot_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ],
    temperature=0,
)

print(zero_shot_response.choices[0].message.content)

<sentiment>
  <label>mixed</label>
  <confidence>0.8</confidence>
</sentiment>


## 2. One-Shot Prompting

**One-shot** means the model receives one example of an input and the expected output, followed by the same question.

In [3]:
one_shot_example_input = "I absolutely loved the new restaurant downtown!"
one_shot_example_output = """<sentiment>
  <label>positive</label>
  <confidence>0.95</confidence>
</sentiment>"""

one_shot_input = f"""Example input:
{one_shot_example_input}

Example output:
{one_shot_example_output}

Now classify this input:
{user_query}

Output:"""

one_shot_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": one_shot_input},
    ],
    temperature=0,
)

print(one_shot_response.choices[0].message.content)

<sentiment>
  <label>mixed</label>
  <confidence>0.80</confidence>
</sentiment>


## 3. Few-Shot Prompting

**Few-shot** means the model receives multiple examples before answering the same question. This section uses three examples.

In [4]:
examples = [
    (
        "I absolutely loved the new restaurant downtown!",
        """<sentiment>
  <label>positive</label>
  <confidence>0.95</confidence>
</sentiment>""",
    ),
    (
        "The movie was okay, nothing special.",
        """<sentiment>
  <label>neutral</label>
  <confidence>0.80</confidence>
</sentiment>""",
    ),
    (
        "Worst customer service I have ever experienced.",
        """<sentiment>
  <label>negative</label>
  <confidence>0.97</confidence>
</sentiment>""",
    ),
]

examples_text = "\n\n".join(
    f"Example input:\n{example_input}\n\nExample output:\n{example_output}"
    for example_input, example_output in examples
)

few_shot_input = f"""{examples_text}

Now classify this input:
{user_query}

Output:"""

few_shot_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": few_shot_input},
    ],
    temperature=0,
)

print(few_shot_response.choices[0].message.content)

<sentiment>
  <label>mixed</label>
  <confidence>0.85</confidence>
</sentiment>


## Quick Revision

| Technique | Number of examples | Prompt contains |
|---|---:|---|
| Zero-shot | 0 | Instructions + question |
| One-shot | 1 | Instructions + one example + question |
| Few-shot | 2 or more | Instructions + multiple examples + question |

**Comparison rule:** The question and model stay the same; only the number of examples changes.

**Note:** The confidence value is generated by the model and is not necessarily a calibrated probability.

### If the model name is unavailable
Run this optional cell to list models visible to your API key, then set `MODEL` above to a suitable text-generation model ID from the output.

In [5]:
available_models = client.models.list()

for model in available_models.data:
    print(model.id)

canopylabs/orpheus-v1-english
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
allam-2-7b
openai/gpt-oss-20b
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
